In [ ]:
!pip install networkx scikit-learn faiss-gpu transformers datasets bitsandbytes gensim node2vec requests

In [ ]:
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
from node2vec import Node2Vec
import numpy as np
import random
import requests
import tqdm

# --------------------------------------
# Step 1: Knowledge Graph Embedding
# --------------------------------------

# Load and parse Hetionet JSON
import json

# Example: download Hetionet JSON file)
url = "https://figshare.com/ndownloader/files/25484210"

response = requests.get(url)

if response.status_code == 200:
  with open("hetionet-v1.0.json", "wb") as f:
      f.write(response.content)
  print("Hetionet JSON file downloaded successfully.")
else:
  print("Failed to download the file. Check the URL or your internet connection.")


with open("hetionet-v1.0.json", "r") as f:
    data = json.load(f)



In [ ]:
# Construct the graph
G = nx.Graph()
for node in data['nodes']:
    G.add_node(node['identifier'], type=node['kind'], name=node['name'])
for edge in data['edges']:
    source = edge['source_id'][1]
    target = edge['target_id'][1]
    edge_type = edge.get('kind', 'unknown')
    G.add_edge(source, target, type=edge_type)

print(f"Graph constructed with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

In [ ]:
import pickle
from tqdm import tqdm

def train_node2vec(graph, dimensions=128, walk_length=5, num_walks=50):
    """
    Train Node2Vec embeddings for graph nodes.
    """
    from node2vec import Node2Vec

    # Train Node2Vec on the graph
    node2vec = Node2Vec(graph, dimensions=dimensions, walk_length=walk_length, num_walks=num_walks, workers=8, p=2.0,  # Return parameter
    q=0.5  )
    model = node2vec.fit(window=10, min_count=1, batch_words=4)
    return model

# Extract a subgraph with a subset of nodes
sub_nodes = list(G.nodes)[:10000]  # Adjust based on memory and time constraints
sub_graph = G.subgraph(sub_nodes)

# Train Node2Vec embeddings for graph nodes
node2vec_model = train_node2vec(sub_graph, walk_length = 5, num_walks=50)
node_embeddings = {node: node2vec_model.wv[node] for node in sub_graph.nodes if node in node2vec_model.wv}


# Save embeddings
with open('node_embeddings.pkl', 'wb') as f:
    pickle.dump(node_embeddings, f)

# Load embeddings later
with open('node_embeddings.pkl', 'rb') as f:
    node_embeddings = pickle.load(f)

In [ ]:
#/content/node_embeddings.pkl

In [ ]:
# --------------------------------------
# Step 2: Text Embedding Using TF-IDF
# --------------------------------------

from datasets import load_dataset

# Load CORD-19 dataset
cord19 = load_dataset("allenai/cord19", name="metadata", split="train[:1000]", trust_remote_code=True)
documents = [doc['abstract'] for doc in cord19 if doc['abstract']]

# Vectorize documents using TF-IDF
vectorizer = TfidfVectorizer(stop_words="english", max_features=10000)
tfidf_matrix = vectorizer.fit_transform(documents)

# Map document indices to titles for easier identification
doc_titles = {idx: doc['title'] for idx, doc in enumerate(cord19) if 'title' in doc}



In [ ]:
# --------------------------------------
# Step 3: Unified Query Retrieval
# --------------------------------------

def hybrid_retrieval(query, graph, tfidf_vectorizer, tfidf_matrix, node_embeddings, top_k=5, graph_hops=2):
    """
    Perform hybrid retrieval by combining graph-based retrieval and TF-IDF.
    """
    # Step 1: Graph Retrieval
    matching_nodes = [
        n for n, attr in graph.nodes(data=True)
        if query.lower() in attr.get('name', '').lower()
    ]
    related_nodes = set()
    for node in matching_nodes:
        related_nodes.update(nx.single_source_shortest_path_length(graph, node, cutoff=graph_hops).keys())

    # Get graph embeddings for retrieved nodes
    graph_vectors = [node_embeddings[node] for node in related_nodes if node in node_embeddings]

    # Step 2: Text Retrieval
    query_vec = tfidf_vectorizer.transform([query])
    text_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_text_indices = text_scores.argsort()[-top_k:][::-1]

    # Combine results: Graph-based + Text-based
    combined_results = []
    if graph_vectors:
        graph_centroid = np.mean(graph_vectors, axis=0)

        # Convert graph_centroid to a string representation before transforming
        graph_centroid_str = ' '.join(map(str, graph_centroid))
        graph_centroid_tfidf = tfidf_vectorizer.transform([graph_centroid_str])  # Project to TF-IDF space
         # Convert graph_centroid_tfidf to a dense array
        graph_centroid_tfidf_dense = graph_centroid_tfidf.toarray()

        for idx in top_text_indices:
            doc_vec = tfidf_matrix[idx].toarray().flatten()
            score = cosine_similarity(graph_centroid_tfidf_dense.reshape(1, -1), doc_vec.reshape(1, -1))[0][0]
            combined_results.append((score, idx))
    else:
        combined_results = [(text_scores[idx], idx) for idx in top_text_indices]

    # Sort by combined scores
    combined_results = sorted(combined_results, key=lambda x: x[0], reverse=True)
    return [(doc_titles[idx], documents[idx]) for _, idx in combined_results[:top_k]]



In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Retrieve and log in with the token directly
login(userdata.get("HF_TOKEN"))

In [ ]:
# --------------------------------------
# Step 4: Generate Answers
# --------------------------------------

from transformers import AutoTokenizer, AutoModelForCausalLM

# Load LLaMA2 model
model_name = "meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=True)
llm = AutoModelForCausalLM.from_pretrained(model_name)

def generate_answer(query, context):
    """
    Generate an answer using the generative model.
    """
    input_text = f"Query: {query}\n\nContext: {context}\n\nAnswer:"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024)
    outputs = llm.generate(**inputs, max_length=2048, temperature=0.7, top_p=0.9)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)



In [ ]:
# --------------------------------------
# Step 5: Query Example
# --------------------------------------

query = "diabetes"
hybrid_results = hybrid_retrieval(query, sub_graph, vectorizer, tfidf_matrix, node_embeddings)
context = "\n".join([doc for _, doc in hybrid_results])

print("Hybrid Retrieval Results:")
for title, doc in hybrid_results:
    print(f"Title: {title}\nAbstract: {doc}\n")

answer = generate_answer(query, context)
print("Generated Answer:", answer)
